In [1]:
import pandas as pd
from pathlib import Path
from mhdr.dataloader.io import read_csv, save_csv
from mhdr.generator.flan_t5_generator import FlanGenerator
INPUT_DIR = Path.cwd() / "input"
OUTPUT_DIR = Path.cwd() / "output"
TEMP_DIR = Path.cwd() / "temp"
from tqdm.auto import tqdm
gen = FlanGenerator()

/Users/haikeyu/Desktop/mentalhealth-dimension-reduction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 558/558 [00:00<00:00, 1085.30it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
questions_df = read_csv(INPUT_DIR / "questions_master.csv")

def build_prompt(seed_list):
    seeds_text = "\n".join([f"- {s}" for s in seed_list])

    return f"""
    Based on the following examples, create a NEW self-report mental health statement.

    Do NOT rewrite or paraphrase any example.
    The new statement must be conceptually related but linguistically distinct.

    Requirements:
    - One clear declarative sentence
    - Natural and realistic
    - Do not copy phrases from the originals

    Original examples:
    {seeds_text}
    """


TARGET_N = 200
BATCH_GEN = 5         
SEED_K = 5            
MAX_TRIES = 2000      


def norm(s: str) -> str:
    return " ".join(s.strip().lower().split())

def is_bad(s: str) -> bool:
    t = s.strip().lower()
    if not s.strip():
        return True
    if t.startswith("how ") or "during the past month" in t:
        return True
    if "?" in s:
        return True
    return False

unique = set()   
rows = []

pbar = tqdm(total=TARGET_N, desc="Generating unique items")

tries = 0
while len(rows) < TARGET_N and tries < MAX_TRIES:
    tries += 1

    sampled_seeds = questions_df["text"].sample(SEED_K).tolist()
    seed_norms = set(norm(x) for x in sampled_seeds)

    prompt = build_prompt(sampled_seeds)

    results = gen.generate(
        prompt,
        num_return_sequences=BATCH_GEN,
        max_new_tokens=30,
        temperature=0.95,
        top_p=0.95,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
    )

    for r in results:
        text = r.strip()

        if is_bad(text):
            continue

        n = norm(text)

        if n in seed_norms:
            continue

        if n in unique:
            continue

        unique.add(n)
        rows.append({
            "text": text,
            "seed_examples": sampled_seeds
        })
        pbar.update(1)

        if len(rows) >= TARGET_N:
            break

pbar.close()

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / "generated_questions.csv", index=False)
print("Done:", df.shape, "tries:", tries)

Generating unique items: 100%|██████████| 200/200 [06:39<00:00,  2.00s/it]

Done: (200, 2) tries: 70
